# MPPP — 03 COLMAP alignment (Navcam, optionally + Mastcam-Z 34 mm)

From PDS products to a COLMAP database and a refined, prior-aligned reconstruction. Pick a named site (`SITES`: Three Forks 684–693, the default test case; Belva 748–815; Rockytop 461–530; add your own) or edit the sol range. With `INCLUDE_ZCAM34 = True` the Mastcam-Z 34 mm frames (`ZL0`/`ZR0`, `_034`) of the same sols join the block: one COLMAP camera per eye (`ZL034`, `ZR034`), initialised from the median label CAHVOR model (or the rough `m20_cmods` XML) and refined like the Navcam cameras. Mastcam-Z left and right exposures have different clocks, so they are separate frames without a rig constraint (experimental). Focus breathing: each eye is split into cameras by focus motor count (`ZCAM_FOCUS_BIN = 30` counts per bin, e.g. `ZL034_F02312`), and each bin refines its own focal length; section 7 plots focal length versus focus.

1. **Select**: Navcam `NLF`/`NRF`, NCAM sequences, one product per exposure (the largest, then the highest version).
2. **Process** with MPPP: 8-bit linear PNG, masks, calibrated intrinsics, CAHV + waypoint poses, frames padded to the detector frame.
3. **Project** (`SfmProject`): one COLMAP camera per eye at full resolution from `mppp/data/m20_cmods/M2020_N{L,R}0_frame.xml` (b1 = b2 = 0; p1, p2 from the XML and refined with `TANGENTIAL = "refine"`, or set to 0 with `"zero"`); half- and quarter-resolution keypoints are scaled to it. One rig: left is the reference, the right camera's offset is the median CAHV left→right pose.
4. **Features** at native resolution with masks → `features.db` (reused if its recorded settings match, e.g. `MAX_NUM_FEATURES`). **Database** → `database.db`, rebuilt on every run (cameras, rig, frames = exposures, full-resolution keypoints, waypoint position priors).
5. **Matching**: `exhaustive` (baseline; use a GPU) or `prior_pairs` (only pairs the priors say overlap; quick / CPU).
6. **Reconstruct**: triangulation from the CAHV poses (two-view tie points kept), then a weighted bundle adjustment (σ in native pixels, Cauchy loss, waypoint priors). Refined: f, principal point and k1–k3 per camera, the right camera's rig orientation, the frame poses; the CAHV stereo baseline is held (it sets the scale).
7. **Alignment health**: tie-point population, reprojection error, camera-model change, stereo-rig stability, pose change against the priors. **Check this before using the result.**
8. Export `error_input/` (native-pixel text model, stations, poses, residuals) — and, under development, the error analysis.

Needs pip `pycolmap` 4.2 and `pyceres` 2.6 in this kernel (`pip install -e .[sfm]`).

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows/conda: duplicate Intel OpenMP guard; before torch
import sys, json
from pathlib import Path
# use the MPPP source this notebook belongs to (the folder above it that holds src/mppp), else an installed mppp
_here = Path.cwd().resolve()
_root = next((p for p in (_here, *_here.parents) if (p / "src" / "mppp").is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root / "src"))
import mppp
print("mppp", mppp.__version__, "from", Path(mppp.__file__).parent)
import pycolmap, pyceres
from mppp.select import find_imgs
from mppp.sfm import (SfmProject, select_best_products, extract_features, build_database, prior_overlap_pairs,
                      match, reconstruct, export_for_error, check_gpu_python, check_ba_environment)
from mppp.sfm.health import assess_alignment, health_table, write_health
print("MPPP", mppp.__version__, "| pycolmap", pycolmap.__version__, "(CUDA:", pycolmap.has_cuda, ") | pyceres", pyceres.__version__)
check_ba_environment()        # stops here if this kernel cannot run the weighted BA (e.g. it is the conda CUDA env)

# Optional GPU for feature extraction and matching (Windows): the python.exe of a second environment with the
# conda-forge CUDA pycolmap of the same COLMAP version (README, "GPU on Windows").  None = run here on the CPU.
GPU_PY = None                 # None: this kernel's pycolmap (CPU wheel on Windows); e.g. r"C:\Users\<you>\AppData\Local\miniconda3\envs\mppp_gpu\python.exe"
if GPU_PY:
    print("GPU env:", check_gpu_python(GPU_PY))

## 1–2. Select and process

In [ ]:
PDS_DIR  = Path("D:/data/m2020")

# named test sites: sol range of the Navcam (+ Mastcam-Z) block; add your own
SITES = {
    "belva":      (748, 815),
    "rockytop":   (461, 530),
    "threeforks": (684, 693),
}
SITE = "threeforks"      # default test case
SOLS = SITES[SITE]

INCLUDE_ZCAM34  = False        # True: add the Mastcam-Z 34 mm frames (ZL0/ZR0 "_034") of the same sols
ZCAM_INTRINSICS = "label"      # initial Mastcam-Z cameras: "label" (median of the label CAHVOR models) or "xml"
ZCAM_FOCUS_BIN  = 30           # Mastcam-Z: one camera per eye, zoom and focus bin of <= 30 motor counts (None: one per eye)
ZCAM_BIN_REFINE = "focal"      # a focus-bin camera refines only fx, fy ("all": also principal point and distortion)
ZCAM_RIG        = False        # no stereo rig for Mastcam-Z: each image its own frame (pairs disagree by up to ~0.08 deg)
# processed images + COLMAP project go here (a separate folder when Mastcam-Z is included)
WORK = Path("D:/scapes") / (f"{SITE}_colmap_nav_zcam34" if INCLUDE_ZCAM34 else f"{SITE}_colmap")

paths = find_imgs(PDS_DIR, ["NLF", "NRF"], sol_range=SOLS)
prefixes = ("NCAM",)
if INCLUDE_ZCAM34:
    paths += find_imgs(PDS_DIR, ["ZL0", "ZR0"], sol_range=SOLS, sequ_id="_034")
    prefixes += ("ZCAM",)
paths, sel = select_best_products(paths, sequence_prefix=prefixes)      # one product per exposure
print(SITE, SOLS, "->", WORK)
print(json.dumps(sel, indent=1)); print(len(paths), "products")

In [ ]:
cfg = mppp.default_config()
cfg["export"].update(formats=["PNG8"], store_mask_in_alpha=False, write_mask_files=True, write_colmap=False)
# mask model: the released "mppp_mask_v1" by default; or cfg["masking"]["checkpoint"] = "<path to your checkpoint>"
PROCESSED = WORK / "processed"
# Frames unsuitable for the analysis: after a first run, delete them from PROCESSED / "images_png8" and set
# KEEP_ONLY_REMAINING = True: only the images left there are processed again, the manifest is rebuilt from them,
# and so is the COLMAP project below (nothing is deleted).
KEEP_ONLY_REMAINING = False
manifest_path = PROCESSED / f"mppp_manifest_{mppp.VERSION_TAG}.json"
if manifest_path.is_file() and not KEEP_ONLY_REMAINING:
    manifest = json.loads(manifest_path.read_text())
else:
    manifest = mppp.process_images(paths, PROCESSED, cfg, mppp.load_waypoints(), provenance=sel,
                                   only_existing="PNG8" if KEEP_ONLY_REMAINING else None)
print(manifest["n_processed"], "processed,", len(manifest["failed"]), "failed")
if manifest.get("only_existing"):
    print("kept", manifest["only_existing"]["n_kept"], "of", manifest["only_existing"]["n_selected"], "selected;",
          manifest["only_existing"]["n_removed"], "removed")

## 3–4. Project, features, database

In [ ]:
from collections import Counter
import numpy as np
PROJECT = WORK / "colmap"
# Tangential distortion p1, p2 (FULL_OPENCV):
#   "refine": start from the Metashape calibration (NL0: P1 = 1.72e-4, P2 = 1.70e-4) and refine them in the BA
#   "xml":    start from the calibration and hold them;   "zero": p1 = p2 = 0, held (MPPP <= 0.14.2)
TANGENTIAL = "refine"
ZERO_TERMS = ("p1", "p2", "b1", "b2") if TANGENTIAL == "zero" else ("b1", "b2")
proj = SfmProject.load(PROJECT) if (PROJECT / "project.json").is_file() else None
wanted = {"zero_terms": list(ZERO_TERMS), "zcam_intrinsics": ZCAM_INTRINSICS,
          "zcam_focus_bin": float(ZCAM_FOCUS_BIN) if ZCAM_FOCUS_BIN else None, "zcam_bin_refine": ZCAM_BIN_REFINE,
          "zcam_rig": ZCAM_RIG, "prior_R_corrected": True}
if (proj is None or KEEP_ONLY_REMAINING                                     # rebuilt from the new manifest
        or any(proj.settings.get(k) != v for k, v in wanted.items())):     # or different camera settings
    proj = SfmProject.create(manifest["images"], PROCESSED, PROJECT, prior_sigma_m=(1.0, 1.0, 1.0),
                            zcam_intrinsics=ZCAM_INTRINSICS, zero_terms=ZERO_TERMS,
                            zcam_focus_bin=ZCAM_FOCUS_BIN, zcam_bin_refine=ZCAM_BIN_REFINE, zcam_rig=ZCAM_RIG)
print(len(proj.images), "images;", {k: round(v["baseline_m"], 4) for k, v in proj.rig.items()}, "m baseline;",
      {k: v["n_pairs"] for k, v in proj.rig.items()}, "CAHV pairs")
for k, c in proj.cameras.items():
    if not c.get("group"):
        print(k, [round(x, 3) for x in c["params"][:5]], f"p1 {c['params'][6]:.2e} p2 {c['params'][7]:.2e}")
bins = Counter(c["group"] for c in proj.cameras.values() if c.get("group"))
for g, n in sorted(bins.items()):
    cs = [c for c in proj.cameras.values() if c.get("group") == g]
    print(f"{g}: {n} focus bins, counts {min(c['focus_count_range'][0] for c in cs):.0f}-"
          f"{max(c['focus_count_range'][1] for c in cs):.0f}, images per bin {sorted(c['n_images'] for c in cs)}")
corr = [r["prior_R_correction_deg"] for r in proj.images if "prior_R_correction_deg" in r]
if corr:   # Mastcam-Z label attitudes rotated to fit the camera's principal point (v0p14.5)
    print(f"Mastcam-Z prior attitude correction: median {np.median(corr):.3f} deg, max {max(corr):.3f} deg ({len(corr)} images)")
print("images per camera:", dict(Counter(r["instrument"] for r in proj.images)))

In [ ]:
# SIFT at native resolution with masks -> features.db.  Reused on reruns only if features.json beside it records
# the same settings (so changing MAX_NUM_FEATURES extracts again); overwrite=True always redoes it.
# COLMAP GUI/CLI alternative: colmap feature_extractor --database_path features.db --image_path images
#     --ImageReader.mask_path masks --SiftExtraction.max_num_features 16380   (check option names with -h)
MAX_NUM_FEATURES = 16380
extract_features(proj, max_num_features=MAX_NUM_FEATURES, use_gpu=True if GPU_PY else None, python=GPU_PY)
print(build_database(proj))          # replaces database.db (and any matches in it) on every run


## 5. Matching

In [ ]:
MATCH_MODE = "exhaustive"            # baseline; "prior_pairs" for a quick / CPU run
if MATCH_MODE == "prior_pairs":
    pairs = prior_overlap_pairs(proj, min_overlap=0.05)
    print(len(pairs), "prior pairs")
print(match(proj, mode=MATCH_MODE, use_gpu=True if GPU_PY else None, python=GPU_PY))

## 6. Reconstruct and refine

In [ ]:
# Rounds of (triangulation threshold [full-res px], Cauchy scale [sigma], max residual kept [native px]),
# then a final BA with the last round's values.  Round 4 repeats round 3 (v0p14.5): does one more pass change anything?
SCHEDULE = ((24.0, 10.0, 8.0), (12.0, 2.0, 4.0), (8.0, 2.0, 2.0), (8.0, 2.0, 2.0))
rec = reconstruct(proj, sigma_px=0.5,           # keypoint sigma in native pixels
                  schedule=SCHEDULE, refine_rig="rotation",
                  refine_tangential=(TANGENTIAL == "refine"),   # p1, p2 free (FULL_OPENCV)
                  min_track_length=2,           # two-view tie points (e.g. seen only by one stereo pair) are kept
                  min_tri_angle_deg=0.5)        # 0.5 deg keeps stereo-only points out to ~49 m at the 0.42 m baseline
summary = export_for_error(proj, rec)
print(json.dumps({k: summary[k] for k in ("tracks", "residual_rms_native_px", "registered_images", "project_images",
                                           "prior_offset_median_m", "station_components")}, indent=1))
for k, c in proj.cameras.items():
    if c.get("group"):
        continue                                  # Mastcam-Z focus bins: see section 7b
    cid = [r["camera_id"] for r in proj.images if r["instrument"] == k and "camera_id" in r][0]
    p0, p1 = c["params"], rec.cameras[cid].params
    print(k, " ".join(f"{n} {a:.4g}->{b:.4g}" for n, a, b in zip(("fx", "fy", "cx", "cy", "k1", "k2", "p1", "p2", "k3"),
                                                               p0[:9], p1[:9])))
print("COLMAP GUI: double-click", proj.root / "open_in_colmap.bat",
      "(native-pixel copy gui_native: keypoints and tie points line up with the images;",
      "open_in_colmap_fullres.bat = the full-resolution project the BA works on)")


## 7. Alignment health — check before any analysis
Each check has provisional warn/fail thresholds (set a priori, not validated limits; pass `thresholds={...}` to change them). The report is written to `<project>/health/` (`health.json`, `health.md`, `health.png`).

| section | checks |
|---|---|
| registration | images registered; frames held at their prior (< 30 observations) |
| tie points | points, track-length population (two-view share), observations per image, cross-station ties, tied station blocks |
| reprojection | median / 95th percentile residual (native px), left vs right eye, outlier images, residual growth towards the image edge |
| camera model | change of f, principal point, and the image displacement of the same ray |
| stereo rig | right-camera rotation vs the CAHV rig; spread of the CAHV pairs |
| poses | refined − prior centre per station, spread within a station, attitude change, scale vs the priors |

In [ ]:
report = assess_alignment(proj, rec)
print(health_table(report))
files = write_health(report, proj.root / "health", rec, proj)
from IPython.display import Image, display
display(Image(filename=files["png"]))
if report["verdict"] == "fail":
    print("\nFAIL: fix the alignment (matching, priors, masks, thresholds) before any error analysis.")

### 7b. Mastcam-Z focus breathing
One camera per eye, zoom and focus bin: the refined focal length of each bin against its focus motor count, with the per-image label focal lengths (grey) and linear fits (refined bins weighted by observations; bins with < 100 observations are crosses and not fitted). Written to `<project>/health/zcam_focus_breathing.{png,csv,json}`.

In [ ]:
from mppp.sfm.zcam import write_focus_breathing
if any(c.get("group") for c in proj.cameras.values()):
    fb = write_focus_breathing(proj, rec)
    for g, f in fb["fits"].items():
        for kind in ("refined", "label"):
            if f[kind]:
                print(f"{g} {kind:7s}: f = {f[kind]['f0_px']:.1f} px at count {f['reference_count']:.0f}, "
                      f"slope {f[kind]['slope_px_per_count']:+.4f} px/count ({f[kind]['slope_pct_per_1000_counts']:+.2f} %/1000 counts), "
                      f"rms {f[kind]['rms_px']:.2f} px, n = {f[kind]['n']}")
        print(f"{g}: {f['bins_fitted']} of {f['bins']} bins fitted")
    display(Image(filename=fb["png"]))
else:
    print("no Mastcam-Z focus-bin cameras (INCLUDE_ZCAM34 = False or ZCAM_FOCUS_BIN = None)")


## 8. Error analysis (`mppp.error`, under development)
The camera-shift plot below is a diagnostic and always runs; the error model is only meaningful for an alignment that passes the health checks.

In [ ]:
# How far each camera moved from its reference (CAHV + waypoint prior): top-down, one arrow per image
# per station (labelled Sol0686 S032D1184), horizontal shift exaggerated, colour = vertical shift;
# plus the station overview and every camera's total shift and attitude change.
import numpy as np, csv
import matplotlib.pyplot as plt
from mppp.sfm.export import plot_camera_shifts
pose_rows = list(csv.DictReader(open(proj.root / "error_input" / "poses.csv")))
fig = plot_camera_shifts(proj, rows=pose_rows, out_png=proj.root / "error_input" / "camera_shifts.png")
plt.show()


In [ ]:
assert report["verdict"] != "fail", "alignment health is FAIL - see section 7"
from mppp.error.colmap import read_colmap, calibrate_eps_from_residuals
model = read_colmap(str(proj.root / "error_input" / "native"))
stations = {r["name"]: r["station"] for r in csv.DictReader(open(proj.root / "error_input" / "stations.csv"))}
model.assign_stations(stations)
print(calibrate_eps_from_residuals(model))